In [2]:
import numpy as np
import networkx as nx


%load_ext autoreload
%autoreload 2

In [3]:
graphs_er = np.load("../Tutorials/ER/resolution-1/graphs-1_p-0.4.npy", allow_pickle=True)
graphs_er = [g[0] for g in graphs_er]
graphs_er_110_160 = np.load("graphs/erdos-renyi_110-160/erdos-renyi_110-160.npy", allow_pickle=True)
g_er_169 = np.load("graphs/erdos-renyi_110-160/erdos-renyi_169.npy", allow_pickle=True)
graphs_er = graphs_er + graphs_er_110_160.tolist() + g_er_169.tolist()

graphs_pwl = np.load("../Tutorials/Power-law/resolution-1/graphs-1_m-1_p-0.1.npy", allow_pickle=True)
graphs_pwl = [g[0] for g in graphs_pwl]
graphs_pwl_110_160 = np.load("graphs/powerlaw_110-160/powerlaw_graphs_110-160.npy", allow_pickle=True)
g_pwl_169 = np.load("graphs/powerlaw_110-160/powerlaw_graph_169.npy", allow_pickle=True)
graphs_pwl = graphs_pwl + graphs_pwl_110_160.tolist() + g_pwl_169.tolist()

In [4]:
results_dir_er = f"ER/resolution-1/graphs-1_p-0.4"
results_dir_pwl = f"Power-law/resolution-1/graphs-1_m-1_p-0.1"

# 169 nodes

## Erdos-Renyi

In [5]:
from Qommunity.searchers.utils import HierarchicalRunMetadata
from iterative_utils import IterativeSearchGraphResults


graphs_results_er = {}
g_size = 169

graphs_sizes = [int(g.number_of_nodes()) for g in graphs_er]

path_original = results_dir_er + f"/graph_size={g_size}"
path_100 = results_dir_er + f"/graph_size={g_size}-extra-100"

communities_list = []
modularities_list = []
times_list = []
div_mods_list = []
div_trees_list = []


for p in [path_100]:
    communities_list.append(np.load(f"{p}/_communities.npy", allow_pickle=True))
    modularities_list.append(np.load(f"{p}/_modularities.npy", allow_pickle=True))
    times_list.append(np.load(f"{p}/_times.npy", allow_pickle=True))
    div_mods_list.append(np.load(f"{p}/_division_modularities.npy", allow_pickle=True))
    div_trees_list.append(np.load(f"{p}/_division_trees.npy", allow_pickle=True))


communities = np.concatenate(communities_list, axis=0)
modularities = np.concatenate(modularities_list, axis=0)
times = np.concatenate(times_list, axis=0)
division_modularities = np.concatenate(div_mods_list, axis=0)
division_trees = np.concatenate(div_trees_list, axis=0)


hierarchical_metadatas = []

# 100 runs
for iter in range(100):
    base_filename = f"{path_100}/_iter_{iter}"
    hm = HierarchicalRunMetadata.load_from_files(base_filename)
    hierarchical_metadatas.append(hm)

iterative_graph_results = IterativeSearchGraphResults(
    graph_ref = graphs_er[graphs_sizes.index(g_size)],
    graph_size=g_size,
    communities = communities,
    modularities = modularities,
    times = times,
    division_modularities = division_modularities,
    division_trees = division_trees,
    hierarchical_metadatas=hierarchical_metadatas
)


graphs_results_er[g_size] = iterative_graph_results

## Power-law

In [6]:
from Qommunity.searchers.utils import HierarchicalRunMetadata
from iterative_utils import IterativeSearchGraphResults


graphs_sizes = [g.number_of_nodes() for g in graphs_er]
graphs_results_pwl = {}
g_size = 169
num_runs = 20

path_original = results_dir_pwl + f"/graph_size={g_size}"
path_10 = results_dir_pwl + f"/graph_size={g_size}-extra-10"
path_20 = results_dir_pwl + f"/graph_size={g_size}-extra-20"
path_20_2 = results_dir_pwl + f"/graph_size={g_size}-extra-20_2"
path_50 = results_dir_pwl + f"/graph_size={g_size}-extra-50"

paths = [path_10, path_20, path_20_2]


hierarchical_metadatas = []
path = results_dir_er + f"/graph_size={g_size}"
path_original = path

# 10 runs
for iter in range(10):
    base_filename = f"{path_10}/_iter_{iter}"
    hm = HierarchicalRunMetadata.load_from_files(base_filename)
    hierarchical_metadatas.append(hm)

# 20 runs
for iter in range(20):
    base_filename = f"{path_20}/_iter_{iter}"
    hm = HierarchicalRunMetadata.load_from_files(base_filename)
    hierarchical_metadatas.append(hm)

# 20 runs extra
for iter in range(20):
    base_filename = f"{path_20_2}/_iter_{iter}"
    hm = HierarchicalRunMetadata.load_from_files(base_filename)
    hierarchical_metadatas.append(hm)

communities_list = []
modularities_list = []
times_list = []
div_mods_list = []
div_trees_list = []

for p in paths:
    communities_list.append(np.load(f"{p}/_communities.npy", allow_pickle=True))
    modularities_list.append(np.load(f"{p}/_modularities.npy", allow_pickle=True))
    times_list.append(np.load(f"{p}/_times.npy", allow_pickle=True))
    div_mods_list.append(np.load(f"{p}/_division_modularities.npy", allow_pickle=True))
    div_trees_list.append(np.load(f"{p}/_division_trees.npy", allow_pickle=True))

communities = np.concatenate(communities_list, axis=0)
modularities = np.concatenate(modularities_list, axis=0)
times = np.concatenate(times_list, axis=0)
division_modularities = np.concatenate(div_mods_list, axis=0)
division_trees = np.concatenate(div_trees_list, axis=0)

iterative_graph_results = IterativeSearchGraphResults(
    graph_ref = graphs_er[graphs_sizes.index(g_size)],
    graph_size=g_size,
    communities = communities,
    modularities = modularities,
    times = times,
    division_modularities = division_modularities,
    division_trees = division_trees,
    hierarchical_metadatas=hierarchical_metadatas
)


graphs_results_pwl[g_size] = iterative_graph_results

## Plots

In [9]:
from utils import recover_ordering, recover_info_ordering, plot_tree_extended
from matplotlib import colormaps
import matplotlib.pyplot as plt
import os


g_size = 169
figsize = (12, 8)

out_dir = f"results/ER/figsize={figsize}/plots_gsize_{g_size}"
os.makedirs(out_dir, exist_ok=True)
out_dir_cbf_nonzero = f"results/ER/figsize={figsize}/plots_gsize_{g_size}_cbf_nonzero"
os.makedirs(out_dir_cbf_nonzero, exist_ok=True)


cmap = colormaps.get_cmap("PuBu")

G = graphs_er[graphs_sizes.index(g_size)]

for run_idx in range(100):
    communities = graphs_results_er[g_size].communities[run_idx]
    division_tree = graphs_results_er[g_size].division_trees[run_idx]
    division_mods = graphs_results_er[g_size].division_modularities[run_idx]
    sampleset_metadata = graphs_results_er[g_size].hierarchical_metadatas[run_idx]

    nodes, root = recover_ordering(G=G, division_tree=division_tree)
    nodes_info, root_info = recover_info_ordering(root=root, nodes=nodes, sampleset_metadata=sampleset_metadata)

    plt.figure(figsize=figsize);
    plot_tree_extended(root_info, division_mods, cmap=cmap, figsize=figsize);

    more_than_one_cbf_nonzero = np.where(np.array(sampleset_metadata.chain_break_fraction) > 0)[0].shape[0] > 1
    if more_than_one_cbf_nonzero:
        filename = os.path.join(out_dir_cbf_nonzero, f"tree_run_{run_idx:03d}.svg")
    else:
        filename = os.path.join(out_dir, f"tree_run_{run_idx:03d}.svg")

    # plt.savefig(filename, dpi=300, bbox_inches="tight");
    plt.savefig(filename, format="svg", bbox_inches="tight");
    plt.close();

c:\Users\basia\anaconda3\envs\qomm_env\Lib\site-packages\pygraphviz\agraph.py:1403: RuntimeWarning: Warning: Could not load "C:\Users\basia\anaconda3\envs\qomm_env\Library\bin\gvplugin_pango.dll" - It was found, so perhaps one of its dependents was not.  Try ldd.

  warnings.warn(b"".join(errors).decode(self.encoding), RuntimeWarning)
c:\Users\basia\anaconda3\envs\qomm_env\Lib\site-packages\pygraphviz\agraph.py:1403: RuntimeWarning: Warning: Could not load "C:\Users\basia\anaconda3\envs\qomm_env\Library\bin\gvplugin_pango.dll" - It was found, so perhaps one of its dependents was not.  Try ldd.

  warnings.warn(b"".join(errors).decode(self.encoding), RuntimeWarning)
c:\Users\basia\anaconda3\envs\qomm_env\Lib\site-packages\pygraphviz\agraph.py:1403: RuntimeWarning: Warning: Could not load "C:\Users\basia\anaconda3\envs\qomm_env\Library\bin\gvplugin_pango.dll" - It was found, so perhaps one of its dependents was not.  Try ldd.

  warnings.warn(b"".join(errors).decode(self.encoding), Runti

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

...

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

In [10]:
from utils import recover_ordering, recover_info_ordering, plot_tree_extended
from matplotlib import colormaps
import matplotlib.pyplot as plt
import os


figsize = (39, 16)

out_dir = f"results/PWL/figsize={figsize}/plots_gsize_{g_size}"
os.makedirs(out_dir, exist_ok=True)
out_dir_cbf_nonzero = f"results/PWL/figsize={figsize}/plots_gsize_{g_size}_cbf_nonzero"
os.makedirs(out_dir_cbf_nonzero, exist_ok=True)


cmap = colormaps.get_cmap("PuBu")

g_size = 169
G = graphs_pwl[graphs_sizes.index(g_size)]

for run_idx in range(50):
    communities = graphs_results_pwl[g_size].communities[run_idx]
    division_tree = graphs_results_pwl[g_size].division_trees[run_idx]
    division_mods = graphs_results_pwl[g_size].division_modularities[run_idx]
    sampleset_metadata = graphs_results_pwl[g_size].hierarchical_metadatas[run_idx]

    nodes, root = recover_ordering(G=G, division_tree=division_tree)
    nodes_info, root_info = recover_info_ordering(root=root, nodes=nodes, sampleset_metadata=sampleset_metadata)

    plt.figure(figsize=figsize);
    plot_tree_extended(root_info, division_mods, cmap=cmap, figsize=figsize);

    more_than_one_cbf_nonzero = np.where(np.array(sampleset_metadata.chain_break_fraction) > 0)[0].shape[0] > 1
    if more_than_one_cbf_nonzero:
        filename = os.path.join(out_dir_cbf_nonzero, f"tree_run_{run_idx:03d}.svg")
    else:
        filename = os.path.join(out_dir, f"tree_run_{run_idx:03d}.svg")

    # plt.savefig(filename, dpi=300, bbox_inches="tight");
    plt.savefig(filename, format="svg", bbox_inches="tight");
    plt.close();

c:\Users\basia\anaconda3\envs\qomm_env\Lib\site-packages\pygraphviz\agraph.py:1403: RuntimeWarning: Warning: Could not load "C:\Users\basia\anaconda3\envs\qomm_env\Library\bin\gvplugin_pango.dll" - It was found, so perhaps one of its dependents was not.  Try ldd.

  warnings.warn(b"".join(errors).decode(self.encoding), RuntimeWarning)
c:\Users\basia\anaconda3\envs\qomm_env\Lib\site-packages\pygraphviz\agraph.py:1403: RuntimeWarning: Warning: Could not load "C:\Users\basia\anaconda3\envs\qomm_env\Library\bin\gvplugin_pango.dll" - It was found, so perhaps one of its dependents was not.  Try ldd.

  warnings.warn(b"".join(errors).decode(self.encoding), RuntimeWarning)
c:\Users\basia\anaconda3\envs\qomm_env\Lib\site-packages\pygraphviz\agraph.py:1403: RuntimeWarning: Warning: Could not load "C:\Users\basia\anaconda3\envs\qomm_env\Library\bin\gvplugin_pango.dll" - It was found, so perhaps one of its dependents was not.  Try ldd.

  warnings.warn(b"".join(errors).decode(self.encoding), Runti

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>

...

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>

<Figure size 3900x1600 with 0 Axes>